# rm-optpessimal-personas — Colab runner

Runs the GPU-dependent generation scripts from [puffables/rm-optpessimal-personas](https://github.com/puffables/rm-optpessimal-personas) on Colab compute.

**Before you start:**
1. `Runtime > Change runtime type > GPU`. A T4 (free tier) is enough for everything **except** the two 27B Gemma-2 base models — those need an A100 (Colab Pro/Pro+, `A100 80GB` if offered) or they'll OOM. Cell 6 lets you skip them.
2. Create a HuggingFace token at https://huggingface.co/settings/tokens (read access is enough) and accept the license on each gated model page you need (`google/gemma-*`, `meta-llama/Llama-3.2-3B-Instruct`).
3. This repo is **private**, so cloning it needs a GitHub token too: create a fine-grained personal access token at https://github.com/settings/tokens?type=beta scoped only to the `puffables/rm-optpessimal-personas` repo, with **Contents: Read-only** permission.
4. In the Colab left sidebar, click the key icon ("Secrets") and add two secrets — `HF_TOKEN` and `GH_TOKEN` — with those tokens, toggling "Notebook access" on for both. This keeps both tokens out of the notebook itself.

**Data persistence:** the repo's `data/` folder is checked into git, so a fresh clone already has prior results, and the three checkpointing scripts (everything except `generate_base_model_logprobs.py`) will skip work that's already done. The last cell zips up anything new/changed so you can pull it back into your local clone and commit it — this notebook does not push to GitHub for you.

In [ ]:
# 1. Confirm a GPU is attached
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Clone the repo (fresh each session)
# Private repo, so the clone is authenticated with GH_TOKEN (from Colab secrets).
# The token is only embedded in the URL for the clone itself; the remote is
# rewritten to drop it immediately after so it isn't left sitting in .git/config.
# Uses an absolute path so this cell is safe to re-run mid-session without
# nesting a clone inside itself.
import os
from google.colab import userdata

REPO_PATH = "puffables/rm-optpessimal-personas"
REPO_DIR = "/content/rm-optpessimal-personas"
GH_TOKEN = userdata.get("GH_TOKEN")

if not os.path.exists(REPO_DIR):
    !git clone https://{GH_TOKEN}@github.com/{REPO_PATH}.git {REPO_DIR}
%cd {REPO_DIR}
!git remote set-url origin https://github.com/{REPO_PATH}.git

In [ ]:
# 3. Install requirements
# Colab's preinstalled torch is already CUDA-enabled, so this mainly adds
# transformers (>=4.45, for the rope_scaling fix Llama-3.x needs), accelerate,
# and the smaller deps.
!pip install -q -r requirements.txt

In [ ]:
# 4. HuggingFace auth (reads the HF_TOKEN secret set up in the sidebar)
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [ ]:
# 5. Sanity-check gated model access before burning compute on a script that'll 401 midway through
from huggingface_hub import model_info

for m in ["google/gemma-2b", "google/gemma-2-9b", "google/gemma-2-27b", "meta-llama/Llama-3.2-3B-Instruct"]:
    try:
        model_info(m)
        print(f"OK   {m}")
    except Exception as e:
        print(f"FAIL {m}: {e}")

## 6. Reward model scores
By default `config/reward_models.yaml` has only one *active* (uncommented) model — `Ray2333/GRM-Llama3.2-3B-rewardmodel-ft` (3B) — the rest are commented out. This fits comfortably on a T4.

In [ ]:
!python generate_reward_model_scores.py

## 7. Persona-conditioned reward model scores
Same active model set as above, swept over `config/personas.yaml` x `config/persona_prompts.yaml`. This script's default batch size (1024) was tuned for a 97GB workstation GPU — pass `--batch-size` sized to whatever GPU you got. 128 (matching this model's tuned batch size for the non-persona sweep above) is a safe starting point on a 16GB T4; drop it further if you still OOM.

In [ ]:
!python generate_persona_reward_model_scores.py --batch-size 128

## 8. Persona-conditioned base model logprobs
Uses `config/llama_base_models.yaml`, which lists only `meta-llama/Llama-3.2-3B-Instruct` — fits on a T4.

In [ ]:
!python generate_persona_base_model_logprobs.py

## 9. Base model logprobs (the heavy one)
`config/gemma_base_models.yaml` spans gemma-1/2 from 2B up to **27B**. Unlike the three scripts above, this one has no resume/skip logic — it recomputes and overwrites every model's CSV on every run, so only pick the models you actually need.

- On a **T4/L4** (or free tier): use the filtered config built below, which drops the two 27B variants.
- On an **A100 80GB**: you can pass `--config gemma_base_models.yaml` for the full list.

The 27B rows are already present in `data/base_model_logits/` from a prior run — only rerun them if `config/prompts.yaml` has changed since.

In [ ]:
# Build a Colab-friendly config that excludes the 27B models (not committed to the repo)
import yaml

with open("config/gemma_base_models.yaml") as f:
    all_models = yaml.safe_load(f)

small_models = [m for m in all_models if "27b" not in m["name"]]
with open("config/gemma_base_models_no27b.yaml", "w") as f:
    yaml.safe_dump(small_models, f)

print(f"{len(small_models)}/{len(all_models)} models kept:")
for m in small_models:
    print(" ", m["name"])

In [ ]:
!python generate_base_model_logprobs.py --config gemma_base_models_no27b.yaml

# Full sweep including the 27B models — only run this on an A100 80GB:
# !python generate_base_model_logprobs.py --config gemma_base_models.yaml

## 10. Pull results back out
Zips the `data/` outputs so you can download them and merge into your local clone (`git status` there will show only the files that actually changed).

In [ ]:
from google.colab import files

!zip -qr data_outputs.zip data/reward_model_scores data/persona_reward_model_scores data/persona_base_model_logits data/base_model_logits
files.download("data_outputs.zip")